In [ ]:
import re
from collections import defaultdict, Counter
import math
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm


from sklearn.model_selection import train_test_split

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import DataCollatorForSeq2Seq

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preprocess DS

### Utility Functions

In [ ]:
def preprocess_pubmedqa(example):
    context = " ".join(example["context"]["contexts"])
    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: Answer yes, no, or maybe. Then justify your answer."
    )
    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )
    short_answer = example['final_decision']  # keep for evaluation

    return {
        "input": input_text,
        "output": target_text,
        "short_answer": short_answer
    }


max_input_length = 512
max_output_length = 128

def tokenize_for_t5(example):
    # Encode inputs
    input_enc = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=max_input_length,
    )

    # Encode targets
    target_enc = tokenizer(
        example["output"],   # <-- FIXED
        truncation=True,
        padding="max_length",
        max_length=max_output_length,
    )

    labels = target_enc["input_ids"]
    labels = [
        l if l != tokenizer.pad_token_id else -100
        for l in labels
    ]

    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": labels,
    }


def collate_fn(batch):
    input_ids = torch.tensor([item["input_ids"] for item in batch], dtype=torch.long)
    attention_mask = torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long)
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
dataset = load_dataset("pubmed_qa", "pqa_labeled")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

In [ ]:
dataset["train"].features

{'pubid': Value('int32'),
 'question': Value('string'),
 'context': {'contexts': List(Value('string')),
  'labels': List(Value('string')),
  'meshes': List(Value('string')),
  'reasoning_required_pred': List(Value('string')),
  'reasoning_free_pred': List(Value('string'))},
 'long_answer': Value('string'),
 'final_decision': Value('string')}

In [ ]:
dataset["train"][0]

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [ ]:
dataset = dataset.map(
    preprocess_pubmedqa,
    remove_columns=dataset["train"].column_names
  )

In [ ]:
dataset["train"][0]

{'input': 'Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? Context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and 

In [ ]:
split_datasets = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_data = split_datasets["train"]
val_data = split_datasets["test"]

In [ ]:
train_data = train_data.remove_columns("short_answer")
print(train_data)

Dataset({
    features: ['input', 'output'],
    num_rows: 900
})


In [ ]:
val_data

Dataset({
    features: ['input', 'output', 'short_answer'],
    num_rows: 100
})

In [ ]:
val_short_answers = [ex["short_answer"] for ex in val_data]

In [ ]:
val_data = val_data.remove_columns("short_answer")
print(val_data)

Dataset({
    features: ['input', 'output'],
    num_rows: 100
})


In [ ]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
train_dataset = train_data.map(tokenize_for_t5, remove_columns=train_data.column_names)
val_dataset   = val_data.map(tokenize_for_t5, remove_columns=val_data.column_names)

In [ ]:
train_dataset, val_dataset

(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 900
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 100
 }))

In [ ]:
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
next(iter(train_loader))

{'input_ids': tensor([[11860,    10, 18524,  ...,     0,     0,     0],
         [11860,    10,  3520,  ...,     0,     0,     0],
         [11860,    10,  3520,  ...,     0,     0,     0],
         ...,
         [11860,    10,  6400,  ...,     0,     0,     0],
         [11860,    10,  3387,  ...,     0,     0,     0],
         [11860,    10,    27,  ...,     0,     0,     0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'labels': tensor([[11801,    10,   150,  ...,  -100,  -100,  -100],
         [11801,    10,   150,  ...,  -100,  -100,  -100],
         [11801,    10,  4273,  ...,  -100,  -100,  -100],
         ...,
         [11801,    10,  2087,  ...,  -100,  -100,  -100],
         [11801,    10,  2087,  ...,  -100,  -100,  -100],
         [11801,    10,   150,  ...,  -100,  -100,  -1

# Model Loading

### Utility Functions

In [ ]:
def freeze_all_params(model):
    for p in model.parameters():
        p.requires_grad = False

def apply_lora_to_ffn(model, r=8, alpha=1.0):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            continue  # Only wrap higher-level FFN
        if module.__class__.__name__ == "T5DenseGatedActDense":
            module.wi_0 = LoRALinear(module.wi_0, r=r, alpha=alpha)
            module.wi_1 = LoRALinear(module.wi_1, r=r, alpha=alpha)
            module.wo = LoRALinear(module.wo, r=r, alpha=alpha)

def merge_lora_to_linear(model):
    for name, module in model.named_modules():
        # Only target LoRALinear
        if isinstance(module, LoRALinear):
            # Create a new nn.Linear with the same shape
            new_linear = nn.Linear(
                module.base.in_features,
                module.base.out_features,
                bias=(module.base.bias is not None)
            ).to(module.base.weight.device)

            # Copy base weight + LoRA contribution
            new_linear.weight.data = module.base.weight.data + (module.B.weight @ module.A.weight) * module.scaling

            if module.base.bias is not None:
                new_linear.bias.data = module.base.bias.data

            # Replace LoRALinear in the parent module
            parent = module._modules
            for key, child in parent.items():
                if child is module:
                    parent[key] = new_linear
                    break


class LoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=1.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False  # Freeze base

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        self.A = nn.Linear(in_dim, r, bias=False)
        self.B = nn.Linear(r, out_dim, bias=False)
        self.scaling = alpha / r

        # Initialize LoRA
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

    # Expose weight/bias to satisfy HF generate()
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias


# Temporarily compute effective weights for generation
def enable_lora_forward(model):
    for module in model.modules():
        if isinstance(module, LoRALinear):
            # Save original forward
            module._original_forward = module.forward
            # Override forward to include LoRA contribution
            module.forward = lambda x, m=module: m.base(x) + m.scaling * m.B(m.A(x))

# Restore original forward for training
def disable_lora_forward(model):
    for module in model.modules():
        if isinstance(module, LoRALinear) and hasattr(module, "_original_forward"):
            module.forward = module._original_forward
            del module._original_forward

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
rank = 128
alpha = 256

# Freeze all parameters
freeze_all_params(model)

# Apply LoRA to FFN layers
apply_lora_to_ffn(model, r=rank, alpha=alpha)

# Make only LoRA parameters trainable
for name, param in model.named_parameters():
    if "A.weight" in name or "B.weight" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"Total parameters: {total:,}")
print(f"Trainable parameters (LoRA): {trainable:,}")
print(f"Frozen parameters: {frozen:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Total parameters: 247,577,856
Trainable parameters (LoRA): 247,577,856
Frozen parameters: 0
Trainable %: 100.0000%


In [ ]:
len(val_short_answers)

100

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

num_epochs = 3
batch_size = 8  # adjust if GPU allows

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    loop = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f"Epoch {epoch+1}")

    for step, batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

    # ----------------------------
    # Evaluation: compute short-answer accuracy
    # ----------------------------
    model.eval()
    enable_lora_forward(model)  # temporarily enable LoRA for generation

    correct = 0
    total = 0

    global_idx = 0 # to index val_short_answers

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=128,
                do_sample=False
            )

            # Extract generated texts
            gen_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

            # Compare with short answers
            for text in gen_texts:
                # Use regex to get the word after 'Answer:'
                match = re.search(r"Answer:\s*(yes|no|maybe)", text, re.IGNORECASE)
                pred = match.group(1).lower() if match else ""

                true_label = val_short_answers[global_idx].lower()
                if pred == true_label:
                    correct += 1
                total += 1
                global_idx += 1

    accuracy = correct / total if total > 0 else 0.0
    print(f"Epoch {epoch+1} short-answer accuracy: {accuracy:.4f}")

    disable_lora_forward(model)  # restore training forward

Epoch 1: 100%|██████████| 113/113 [02:21<00:00,  1.25s/it, loss=1.94]


Epoch 1 average loss: 2.3397


Evaluating: 100%|██████████| 13/13 [00:25<00:00,  1.99s/it]


Epoch 1 short-answer accuracy: 0.5000


Epoch 2: 100%|██████████| 113/113 [02:30<00:00,  1.33s/it, loss=1.97]


...

Epoch 2 short-answer accuracy: 0.5500


Epoch 3: 100%|██████████| 113/113 [02:31<00:00,  1.34s/it, loss=1.36]


Epoch 3 average loss: 1.3551


Evaluating: 100%|██████████| 13/13 [00:28<00:00,  2.17s/it]

Epoch 3 short-answer accuracy: 0.5600


In [ ]:
# Enable LoRA forward for generation
enable_lora_forward(model)
model.eval()

generated_texts = []

# raw_val_texts has 'input', raw_val_labels has 'short_answer'
raw_val_texts = [ex["input"] for ex in val_data]
raw_val_labels = [ex["output"] for ex in val_data]  # True short answer

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Generating on val set"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            do_sample=False
        )

        batch_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
        generated_texts.extend(batch_texts)

# Now print generated vs true short answer
for i in range(5):  # first 5 examples
    print(f"Example {i+1}:")
    print(f"Question + Context + Instruction:\n{raw_val_texts[i]}\n")
    print(f"True Short Answer: {raw_val_labels[i]}")
    print(f"Generated Answer:\n{generated_texts[i]}\n")
    print("-" * 80)

disable_lora_forward(model)

Generating on val set: 100%|██████████| 13/13 [00:28<00:00,  2.22s/it]

Example 1:
Question + Context + Instruction:
Question: Is eligibility for a chemotherapy protocol a good prognostic factor for invasive bladder cancer after radical cystectomy? Context: To assess whether eligibility to an adjuvant chemotherapy protocol in itself represents a good prognostic factor after radical cystectomy for bladder cancer. Between April 1984 and May 1989, our institution entered 35 patients with invasive bladder cancer into the Swiss Group for Clinical and Epidemiological Cancer Research (SAKK) study 09/84. They were randomly assigned to either observation or three postoperative courses of cisplatin monotherapy after cystectomy. This study had a negative result. The outcome of these 35 patients (protocol group) was compared with an age- and tumor-stage-matched cohort (matched group; n = 35) who also underwent cystectomy during the same period, but were not entered into the SAKK study, as well as the remaining 57 patients treated during the study period for the same i

In [ ]:
val_data[0]["output"]

'Answer: yes. Explanation: These data suggest that being willing and fit enough for a chemotherapy protocol is a good prognostic factor for invasive bladder cancer. This eligibility bias emphasizes the need for prospective, randomized trials, and indicates that single-group studies using historical or matched controls have to be interpreted with caution.'